#  Análisis exploratorio de datos del dataset 

Se busca analizar el consumo doméstico de café en países productores (1990 - 2020) para identificar tendencias, mercados clave y oportunidades que apoyen la estrategia de exportación de Garden Coffee

**Contenido**
1. Carga y calidad de datos
2. Transformación a formato largo
3. Analisis exploratorio
4. Forecast con rangos
5. Segmentación de mercados
6. Conclusiones y recomendaciones

In [1]:
import sys
from pathlib import Path

PROJECT = Path.cwd().parent / "02_proyecto"
sys.path.append(str(PROJECT))

from src import config
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "plotly_mimetype+notebook"
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.0f}".format)

config.RAW_FILE, config.RAW_FILE.exists()

(WindowsPath('C:/Users/nathm/Proyectos/Garden-Coffee/02_proyecto/data/raw/coffee_db.parquet'),
 True)

In [2]:
df_raw = pd.read_parquet(config.RAW_FILE)
print(df_raw.shape)
df_raw.head()

(55, 33)


,Country,Coffee type,1990/91,1991/92,1992/93,1993/94,1994/95,1995/96,1996/97,1997/98,1998/99,1999/00,2000/01,2001/02,2002/03,2003/04,2004/05,2005/06,2006/07,2007/08,2008/09,2009/10,2010/11,2011/12,2012/13,2013/14,2014/15,2015/16,2016/17,2017/18,2018/19,2019/20,Total_domestic_consumption
0,Angola,Robusta/Arabica,1200000,1800000,2100000,1200000,1500000,600000,1200000,2400000,1800000,1200000,1200000,1200000,1200000,900000,900000,900000,1800000,1800000,1800000,1800000,1800000,1800000,1800000,1800000,1800000,1800000,1800000,1800000,1800000,1800000,46500000
1,Bolivia (Plurinational State of),Arabica,1500000,1620000,1650000,1710000,1770000,1830000,1890000,1950000,1980000,2040000,2100000,2190000,2250000,2310000,2700000,2460000,2520000,2610000,2700000,2760000,2850000,2940000,3030000,3120000,3210000,3300000,3420000,3510000,3600000,3660000,75180000
2,Brazil,Arabica/Robusta,492000000,510000000,534000000,546000000,558000000,606000000,660000000,690000000,732000000,762000000,792000000,815400000,825000000,852000000,896760000,932280000,979860000,1026600000,1059600000,1103400000,1147920000,1183200000,1219800000,1205100000,1219980000,1230480000,1273500000,1319820000,1332000000,1320000000,27824700000
3,Burundi,Arabica/Robusta,120000,96000,102000,114600,120000,120000,120000,120000,120000,120000,120000,120000,120000,120000,120000,114360,93600,81420,86100,83940,120000,120000,120000,120000,120000,120000,120000,120000,120000,120000,3412020
4,Ecuador,Arabica/Robusta,21000000,21000000,21000000,21000000,21000000,21000000,18000000,18000000,18000000,13800000,12000000,12000000,9000000,9000000,9000000,9000000,9000000,9000000,9000000,9000000,9000000,9000000,9000000,9300000,9300000,9300000,9300000,9300000,9300000,8940000,381540000


## Carga y calidad de datos

Antes del análisis se verifica la estructura del dataset, su integridad, las unidades de medida y la presencia de valores que sean como sospechoso

In [3]:
import re

year_cols = [c for c in df_raw.columns if re.fullmatch(r"\d{4}/\d{2}", c)]
id_cols = [c for c in df_raw.columns if c not in year_cols]

print("Columnas de año:", len(year_cols), "|", year_cols[0], "a", year_cols[-1])
print("Otras columnas:", id_cols)

Columnas de año: 30 | 1990/91 a 2019/20
Otras columnas: ['Country', 'Coffee type', 'Total_domestic_consumption']


In [4]:
quality = pd.Series({
    "Filas": len(df_raw),
    "Países únicos": df_raw["Country"].nunique(),
    "Países duplicados": df_raw["Country"].duplicated().sum(),
    "Valores nulos": df_raw.isna().sum().sum(),
    "Valores negativos": (df_raw[year_cols] < 0).sum().sum(),
    "Máx. diferencia total vs suma de años": (df_raw[year_cols].sum(axis=1) - df_raw["Total_domestic_consumption"]).abs().max(),
})
quality.to_frame("Valor")

,Valor
Filas,55
Países únicos,55
Países duplicados,0
Valores nulos,0
Valores negativos,0
Máx. diferencia total vs suma de años,0


In [5]:
df_raw["Coffee type"].value_counts().to_frame("Países")

,Países
Coffee type,
Arabica,22
Robusta,15
Robusta/Arabica,10
Arabica/Robusta,8


**Tipo de café** Cada país aparece una sola vez con una única etiqueta, entonces `Coffee type` es un atributo del país y no un desglose del consumo, no es posible saber cuánto Arábica o Robusta consume cada país, solo comparar grupos de países según su tipo principal

In [6]:
zero_share = (df_raw[year_cols] == 0).mean(axis=1) * 100
ceros = df_raw.loc[zero_share > 0, ["Country"]].assign(pct_anios_en_cero=zero_share[zero_share > 0])
ceros.sort_values("pct_anios_en_cero", ascending=False)

,Country,pct_anios_en_cero
28,Equatorial Guinea,100
42,Nepal,100
12,Timor-Leste,67
54,Yemen,43
39,Lao People's Democratic Republic,40
20,Zambia,37
34,Guyana,27


In [7]:
valores_distintos = df_raw[year_cols].nunique(axis=1)
planas = df_raw.loc[valores_distintos <= 3, ["Country"]].assign(valores_distintos=valores_distintos[valores_distintos <= 3])
planas.sort_values("valores_distintos")

,Country,valores_distintos
28,Equatorial Guinea,1
42,Nepal,1
12,Timor-Leste,2
7,Malawi,2
14,Congo,2
26,Democratic Republic of Congo,2
40,Liberia,2
9,Paraguay,2
25,Côte d'Ivoire,3
44,Nigeria,3


# Conclusiones de calidad

- El dataset esta integro: 55 países, sin nulos, duplicados ni negativos, y el total coincide con la suma anual.
- Las unidades son kilogramos, no tazas.
- `Coffee type` describe el tipo principal de cada país, no todo a detalle del consumo
- Nepal y Guinea Ecuatorial reportan cero en todo el periodo, se interpretan como no reportados y se excluyen del modelado.
- Cinco países tienen ceros parciales y once tienen series casi constantes, lo que sugiere estimaciones repetidas, se conservan en el análisis descriptivo, pero se marcan y se tratan con cautela en el forecast.

## Transformación a formato largo
El dataset de formato ancho a formato largo (una fila por país y año), se añaden banderas de calidad para que los análisis posteriores puedan filtrar con cuidado los datos sospechosos.

In [8]:
df_long = df_raw.melt(
    id_vars=["Country", "Coffee type"],
    value_vars=year_cols,
    var_name="crop_year",
    value_name="consumption_kg",
)
df_long["year"] = df_long["crop_year"].str[:4].astype(int)
df_long = df_long.rename(columns={"Country": "country", "Coffee type": "coffee_type"})

print(df_long.shape)
df_long.head()

(1650, 5)


,country,coffee_type,crop_year,consumption_kg,year
0,Angola,Robusta/Arabica,1990/91,1200000,1990
1,Bolivia (Plurinational State of),Arabica,1990/91,1500000,1990
2,Brazil,Arabica/Robusta,1990/91,492000000,1990
3,Burundi,Arabica/Robusta,1990/91,120000,1990
4,Ecuador,Arabica/Robusta,1990/91,21000000,1990


In [9]:
paises_sin_reporte = df_raw.loc[zero_share == 100, "Country"].tolist()
paises_planos = planas["Country"].tolist()

df_long["reported"] = ~df_long["country"].isin(paises_sin_reporte)
df_long["zero_flag"] = df_long["consumption_kg"].eq(0)
df_long["flat_series"] = df_long["country"].isin(paises_planos)

columnas = ["country", "coffee_type", "year", "crop_year", "consumption_kg", "reported", "zero_flag", "flat_series"]
df_long = df_long[columnas].sort_values(["country", "year"]).reset_index(drop=True)
df_long.head()

,country,coffee_type,year,crop_year,consumption_kg,reported,zero_flag,flat_series
0,Angola,Robusta/Arabica,1990,1990/91,1200000,True,False,False
1,Angola,Robusta/Arabica,1991,1991/92,1800000,True,False,False
2,Angola,Robusta/Arabica,1992,1992/93,2100000,True,False,False
3,Angola,Robusta/Arabica,1993,1993/94,1200000,True,False,False
4,Angola,Robusta/Arabica,1994,1994/95,1500000,True,False,False


In [10]:
checks = pd.Series({
    "Filas esperadas (países x años)": len(df_raw) * len(year_cols),
    "Filas obtenidas": len(df_long),
    "Suma formato ancho": df_raw[year_cols].sum().sum(),
    "Suma formato largo": df_long["consumption_kg"].sum(),
    "Países sin reporte": df_long.loc[~df_long["reported"], "country"].nunique(),
    "Países con serie plana": df_long.loc[df_long["flat_series"], "country"].nunique(),
})
checks.to_frame("Valor")

,Valor
Filas esperadas (países x años),1650
Filas obtenidas,1650
Suma formato ancho,61186765103
Suma formato largo,61186765103
Países sin reporte,2
Países con serie plana,11


In [11]:
df_long.query("country == 'Colombia'").tail()

,country,coffee_type,year,crop_year,consumption_kg,reported,zero_flag,flat_series
205,Colombia,Arabica,2015,2015/16,103222380,True,False,False
206,Colombia,Arabica,2016,2016/17,104186484,True,False,False
207,Colombia,Arabica,2017,2017/18,107596260,True,False,False
208,Colombia,Arabica,2018,2018/19,107484600,True,False,False
209,Colombia,Arabica,2019,2019/20,121486440,True,False,False


In [ ]:
from src import etl

df_check = etl.to_long(etl.load_raw())
pd.testing.assert_frame_equal(df_check, df_long)
print("El modulo src/etl.py reproduce exactamente la transformación del notebook")

El módulo src/etl.py reproduce exactamente la transformación del notebook
